# Helpdesk LoRA
Ative GPU CUDA no Colab. Compacte a pasta helpdesk-ml como helpdesk-ml.zip e envie abaixo. O experimento é pequeno e exploratório. O treinamento ainda não foi executado pelo assistente.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile
uploaded = files.upload()
archive = next((name for name in uploaded if name.endswith(".zip")), None)
if archive is None:
    raise ValueError("Envie helpdesk-ml.zip")
destination = Path("/content/helpdesk_project")
destination.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as bundle:
    for name in bundle.namelist():
        if not (destination/name).resolve().is_relative_to(destination.resolve()):
            raise ValueError("Caminho inválido no ZIP")
    bundle.extractall(destination)
project = next(destination.rglob("train_lora.py")).parent


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(project/"requirements-llm.txt")], check=True)


In [ ]:
subprocess.run([sys.executable, str(project/"train_lora.py")], check=True)


In [ ]:
import json
metrics = json.loads((project/"adapter/metrics.json").read_text())
print({key: metrics[key] for key in ["before_test", "after_test"]})


Revise as métricas antes de publicar. A próxima célula abre um campo protegido para autenticação na sua conta.

In [ ]:
from huggingface_hub import notebook_login, HfApi
notebook_login()


In [ ]:
repo_id = "JuliaNeris0017/helpdesk-lora-pt"
api = HfApi()
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
api.upload_folder(repo_id=repo_id, folder_path=str(project/"adapter"), repo_type="model")
print("https://huggingface.co/" + repo_id)
